# 🚦 Módulo 8: El Guardián de la Puerta — Rate Limiter
## Notebook de Conocimiento Guiado

**Objetivo:** Dominar los cuatro algoritmos clásicos de rate limiting
comprendiendo sus tradeoffs de memoria, CPU y comportamiento bajo ráfagas.

**Lore:** Eres el Guardián de la Puerta Galáctica. Controlas el flujo de
naves que entran al sistema solar. Demasiadas a la vez → colapso del sistema.

| Sección | Concepto |
|---------|---------|
| 📚 Teoría | ¿Por qué rate limiting? Casos de uso |
| 🔨 Demo | Fixed Window, Sliding Window Log |
| 🔨 Demo | Token Bucket, Leaky Bucket |
| ✏️ Ejercicio 1 | Sliding Window Counter (híbrido) |
| ✏️ Ejercicio 2 | Rate limiter distribuido con timestamps |
| 🎯 Quiz | Preguntas de system design |

## 📚 Parte 1: ¿Por qué Rate Limiting?

**Casos de uso reales:**
- **APIs públicas**: GitHub permite 5000 req/hora autenticado, 60 sin autenticar
- **Login**: máximo 5 intentos fallidos por cuenta → previene fuerza bruta
- **SMS/Email**: máximo 10 mensajes por minuto → previene spam
- **Microservicios**: proteger servicios backend de picos de tráfico

**Los 4 algoritmos:**

| Algoritmo | Memoria | Ráfagas | Precisión | Complejidad |
|-----------|---------|---------|-----------|-------------|
| Fixed Window | O(1) | Permite doble spike | Baja | Simple |
| Sliding Window Log | O(n) | No permite | Alta | Media |
| Token Bucket | O(1) | Controla | Alta | Media |
| Leaky Bucket | O(1) | Suaviza | Alta | Media |

In [ ]:
# 🔨 IMPLEMENTACIÓN: Fixed Window Rate Limiter

import time
from collections import defaultdict, deque

class FixedWindowRateLimiter:
    """
    Ventana fija: cuenta requests por ventana de tiempo.
    
    Problema: un cliente puede hacer 2x requests justo en el borde de dos ventanas.
    
    Ejemplo con max=5, ventana=60s:
      t=59s: 5 requests → OK (ventana 1)
      t=61s: 5 requests → OK (ventana 2) ← 10 requests en 2 segundos!
    """
    def __init__(self, max_requests, window_seconds):
        self.max_requests = max_requests
        self.window_seconds = window_seconds
        self._estado = defaultdict(lambda: [0.0, 0])   # [inicio_ventana, contador]
    
    def allow_request(self, client_id="default"):
        now = time.monotonic()
        estado = self._estado[client_id]
        inicio, contador = estado
        
        if now - inicio >= self.window_seconds:
            estado[0] = now
            estado[1] = 1
            return True
        
        if contador < self.max_requests:
            estado[1] += 1
            return True
        
        return False   # Bloqueado

# Demo
rl = FixedWindowRateLimiter(max_requests=3, window_seconds=1.0)
resultados = [rl.allow_request("usuario-1") for _ in range(5)]
print("Fixed Window (3 req/s):", resultados)
# [True, True, True, False, False] — las últimas 2 bloqueadas

In [ ]:
# 🔨 IMPLEMENTACIÓN: Token Bucket

class TokenBucketRateLimiter:
    """
    Bucket de tokens: los tokens se acumulan a una tasa fija.
    Permite ráfagas controladas: si no usaste tokens, se acumulan (hasta max_tokens).
    
    Analogía: Un grifo llena un cubo a velocidad constante.
    Puedes vaciar el cubo de golpe (ráfaga), pero no más que su capacidad.
    """
    def __init__(self, max_tokens, tokens_por_segundo):
        self.max_tokens = max_tokens
        self.tokens_por_segundo = tokens_por_segundo
        self._tokens = defaultdict(lambda: [float(max_tokens), time.monotonic()])
    
    def allow_request(self, client_id="default", tokens_requeridos=1):
        now = time.monotonic()
        estado = self._tokens[client_id]
        tokens_actuales, ultima_recarga = estado
        
        # Recargar tokens según tiempo transcurrido
        elapsed = now - ultima_recarga
        tokens_ganados = elapsed * self.tokens_por_segundo
        tokens_actuales = min(self.max_tokens, tokens_actuales + tokens_ganados)
        estado[1] = now
        
        if tokens_actuales >= tokens_requeridos:
            estado[0] = tokens_actuales - tokens_requeridos
            return True
        
        estado[0] = tokens_actuales
        return False

# Demo
tb = TokenBucketRateLimiter(max_tokens=5, tokens_por_segundo=2.0)

print("Token Bucket (capacidad=5, recarga=2/s):")
print("Ráfaga inicial (5 requests rápidas):")
for i in range(6):
    ok = tb.allow_request("user")
    print(f"  Request {i+1}: {'✓ PERMITIDA' if ok else '✗ BLOQUEADA'}")

In [ ]:
# 🔨 IMPLEMENTACIÓN: Sliding Window Log

class SlidingWindowLogRateLimiter:
    """
    Log deslizante: guarda el timestamp de cada request.
    Siempre mira los últimos window_seconds segundos.
    
    Ventaja: sin el problema del spike en los bordes.
    Desventaja: O(n) memoria donde n = max_requests.
    """
    def __init__(self, max_requests, window_seconds):
        self.max_requests = max_requests
        self.window_seconds = window_seconds
        self._logs = defaultdict(deque)
    
    def allow_request(self, client_id="default"):
        now = time.monotonic()
        log = self._logs[client_id]
        
        # Eliminar requests fuera de la ventana
        limite = now - self.window_seconds
        while log and log[0] <= limite:
            log.popleft()
        
        if len(log) < self.max_requests:
            log.append(now)
            return True
        return False

# Demo comparativo
fixed = FixedWindowRateLimiter(3, 1.0)
sliding = SlidingWindowLogRateLimiter(3, 1.0)

print("Comparación Fixed vs Sliding (3 req/s):")
for i in range(5):
    r_fixed = fixed.allow_request()
    r_sliding = sliding.allow_request()
    print(f"  Request {i+1}: Fixed={'✓' if r_fixed else '✗'} | Sliding={'✓' if r_sliding else '✗'}")

## ✏️ Ejercicio 1: Sliding Window Counter (híbrido)

El **Sliding Window Counter** es un compromiso entre Fixed Window (O(1) memoria)
y Sliding Window Log (exacto pero O(n) memoria):

```
Ventana actual     Ventana anterior
|────────────────|────────────────|
       ▲ now
       
rate ≈ contador_prev × (fracción de ventana prev visible) + contador_actual
```

In [ ]:
# ✏️ EJERCICIO 1

class SlidingWindowCounterRL:
    """
    Aproximación O(1) del sliding window usando dos ventanas consecutivas.
    Estima el conteo del período deslizante con interpolación lineal.
    """
    def __init__(self, max_requests, window_seconds):
        self.max_requests = max_requests
        self.window_seconds = window_seconds
        # {client: {'ventana_inicio': float, 'contador_actual': int, 'contador_prev': int}}
        self._estado = defaultdict(lambda: {'inicio': 0.0, 'actual': 0, 'prev': 0})
    
    def allow_request(self, client_id="default"):
        # TODO: Implementar el algoritmo Sliding Window Counter
        # 1. Calcular cuánto de la ventana anterior es visible
        # 2. rate_estimado = prev * fraccion_visible + actual
        # 3. Si rate_estimado + 1 <= max_requests → permitir
        pass

# Test
swc = SlidingWindowCounterRL(max_requests=5, window_seconds=1.0)
for i in range(7):
    ok = swc.allow_request("u1")
    print(f"Request {i+1}: {'✓' if ok else '✗'}")

In [ ]:
# 💡 SOLUCIÓN — Ejercicio 1

class SlidingWindowCounterRL:
    def __init__(self, max_requests, window_seconds):
        self.max_requests = max_requests
        self.window_seconds = window_seconds
        self._estado = defaultdict(lambda: {'inicio': 0.0, 'actual': 0, 'prev': 0})
    
    def allow_request(self, client_id="default"):
        now = time.monotonic()
        e = self._estado[client_id]
        
        transcurrido = now - e['inicio']
        
        if transcurrido >= self.window_seconds * 2:
            # Ambas ventanas caducaron
            e['inicio'] = now
            e['prev'] = 0
            e['actual'] = 0
        elif transcurrido >= self.window_seconds:
            # Girar ventanas
            e['prev'] = e['actual']
            e['actual'] = 0
            e['inicio'] += self.window_seconds
            transcurrido -= self.window_seconds
        
        fraccion_visible = 1.0 - (transcurrido / self.window_seconds)
        estimado = e['prev'] * fraccion_visible + e['actual']
        
        if estimado + 1 <= self.max_requests:
            e['actual'] += 1
            return True
        return False

swc = SlidingWindowCounterRL(max_requests=5, window_seconds=1.0)
for i in range(7):
    ok = swc.allow_request("u1")
    print(f"Request {i+1}: {'✓ PERMITIDA' if ok else '✗ BLOQUEADA'}")

## ✏️ Ejercicio 2: Rate limiter con múltiples niveles

In [ ]:
# ✏️ EJERCICIO 2

class MultiLevelRateLimiter:
    """
    Rate limiter con múltiples niveles:
    - Nivel 1: max 5 requests/segundo
    - Nivel 2: max 100 requests/minuto
    - Nivel 3: max 1000 requests/hora
    Una request pasa solo si TODOS los niveles la permiten.
    """
    def __init__(self):
        # TODO: Crear tres TokenBucketRateLimiter con los límites arriba descritos
        pass
    
    def allow_request(self, client_id="default"):
        # TODO: Retorna True solo si los 3 niveles permiten la request
        pass

# Test
ml = MultiLevelRateLimiter()
permitidas = sum(1 for _ in range(20) if ml.allow_request("u1"))
print(f"Permitidas de 20 requests rápidas: {permitidas}")

In [ ]:
# 💡 SOLUCIÓN — Ejercicio 2

class MultiLevelRateLimiter:
    def __init__(self):
        self._niveles = [
            TokenBucketRateLimiter(max_tokens=5,    tokens_por_segundo=5.0),
            TokenBucketRateLimiter(max_tokens=100,  tokens_por_segundo=100/60),
            TokenBucketRateLimiter(max_tokens=1000, tokens_por_segundo=1000/3600),
        ]
    
    def allow_request(self, client_id="default"):
        return all(n.allow_request(client_id) for n in self._niveles)

ml = MultiLevelRateLimiter()
permitidas = sum(1 for _ in range(20) if ml.allow_request("u1"))
print(f"Permitidas de 20 requests rápidas: {permitidas}")
print("(Limitadas por el nivel de 5/s → máximo 5 en ráfaga inicial)")

## 🎯 Quiz — Preguntas de Entrevista

**P1:** ¿Qué algoritmo usarías para limitar requests en una API pública como GitHub?
> **R:** Token Bucket — permite ráfagas cortas (útil para scripts), tiene O(1) memoria,
> y los tokens se acumulan cuando el usuario no hace requests.

**P2:** ¿Cómo implementarías rate limiting en un sistema distribuido con 50 servidores?
> **R:** Redis con operaciones atómicas (INCR + EXPIRE para fixed window,
> sorted sets para sliding window). Redis Cluster garantiza consistencia y baja latencia.

**P3:** ¿Cuál es el "thundering herd problem" y cómo lo causa el rate limiter?
> **R:** Cuando muchos clientes se bloquean simultáneamente y todos reintentan al mismo tiempo
> cuando la ventana se reinicia. Solución: jitter aleatorio en los reintentos, exponential backoff.

**P4:** ¿Por qué el Sliding Window Counter (aprox.) es preferible en producción al Log exacto?
> **R:** El Log guarda O(max_requests) timestamps por cliente. Con millones de usuarios y
> límites de 1000 req/hora, la memoria sería prohibitiva. El Counter usa O(1) con ~10% error.